# 02 — Lote de reels (A100)

Genera el lote completo. **Cada vídeo queda en Drive** junto a su `.story.json` y un `manifest.json`.

El notebook comprueba el doctor antes de empezar y avisa si falta el informe de verificación del notebook 01.

Ajusta la celda 1 para decidir cuántos reels quieres, la duración y las escenas: esas tres cifras son las que mandan en el consumo de unidades.


In [ ]:
#@title 1. Configuración del lote { display-mode: "form" }
#@markdown Número de reels a generar:
COUNT = 5 #@param {type:"integer"}
#@markdown Duración objetivo por reel (minutos):
DURATION = 6.0 #@param {type:"number"}
#@markdown Escenas por reel (una imagen SDXL cada una):
SCENES = 10 #@param {type:"integer"}
#@markdown Temas, uno por vídeo y en orden:
THEMES = "amor_fati,memento_mori,control_dichotomy,inner_fortress,present_moment" #@param {type:"string"}
#@markdown Usar un LLM para el guion si hay API key:
USE_LLM = True #@param {type:"boolean"}
#@markdown Efectos cinematográficos y motor visual (partículas):
QUALITY = True #@param {type:"boolean"}

import time

THEME_LIST = [theme.strip() for theme in THEMES.split(",") if theme.strip()]
RUN_NAME = "lote_" + time.strftime("%Y%m%d_%H%M")

print("lote     :", RUN_NAME)
print("videos   :", COUNT)
print("duracion :", DURATION, "min |", SCENES, "escenas")
print("temas    :", THEME_LIST)


In [ ]:
#@title 2. Preflight — doctor + última verificación { display-mode: "form" }
import json
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/canal-estoicismo")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import scripts.colab_env as env

if not (env.PROJECT_DIR / ".git").exists():
    raise SystemExit("Sesión nueva: ejecuta primero 00_entorno_A100.ipynb.")
if not (env.PROJECT_DIR / "output").is_symlink():
    raise SystemExit("output/ no apunta a Drive: vuelve a ejecutar 00_entorno_A100.ipynb.")

code = env.run(
    [
        sys.executable, "main.py", "doctor",
        "--format", "reel",
        "--profile", "auto",
        "--image-provider", "sdxl_local",
    ],
    cwd=env.PROJECT_DIR,
)
env.gate("doctor", code == 0, "no lances el lote sin el doctor en PASS")

report_path = env.PROJECT_DIR / "output" / "_checks" / "acceptance.json"
if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    verdict = "OK" if report.get("passed") else "FALLÓ"
    print(f"\nÚltima verificación: {report.get('created_at')} | {report.get('device')} | {verdict}")
else:
    print("\n[AVISO] No hay informe de verificación.")
    print("        Ejecuta 01_verificacion_A100.ipynb antes de un lote largo.")


In [ ]:
#@title 3. Ejecutar el lote { display-mode: "form" }
output_dir = env.DRIVE_ROOT / "output" / RUN_NAME
output_dir.mkdir(parents=True, exist_ok=True)
print("destino:", output_dir)

command = [
    sys.executable, "main.py", "batch",
    "--count", str(COUNT),
    "--output-dir", str(output_dir),
    "--themes", *THEME_LIST,
    "--duration", str(DURATION),
    "--scenes", str(SCENES),
    "--format", "reel",
    "--profile", "auto",
    "--image-provider", "sdxl_local",
]
if USE_LLM:
    command += ["--llm", "auto"]
if QUALITY:
    command += ["--cinematic", "--visual-engine"]

code = env.run(command, cwd=env.PROJECT_DIR)
env.gate("lote completado", code == 0, "mira los [FAIL] por vídeo en el log de arriba")


In [ ]:
#@title 4. Verificar los vídeos y guardar el manifest { display-mode: "form" }
rows = env.verify_videos(output_dir, "reel")
env.gate("se generó al menos un vídeo", len(rows) > 0, "el lote no produjo ningún .mp4")

manifest = env.write_manifest(
    output_dir,
    rows,
    extra={
        "run": RUN_NAME,
        "count_requested": COUNT,
        "duration_min": DURATION,
        "scenes": SCENES,
        "themes": THEME_LIST,
        "with_llm": USE_LLM,
    },
)

good = [row for row in rows if row["has_audio"] and row["duration"] > 0]
print(f"\n[OK] {len(good)}/{len(rows)} reels verificados.")
for row in rows:
    print(f"  {row['name']}  {row['duration']:.1f}s  {row['size_mb']:.1f} MB")


## Siguiente paso

Abre **`03_entrega.ipynb`** para verificar el lote desde Drive, descargarlo en un ZIP y reanudar cualquier reel que se haya cortado.

### Si la sesión se cae a mitad

No pasa nada grave: las imágenes ya generadas están cacheadas en Drive.

1. Vuelve a ejecutar `00_entorno_A100.ipynb` (celdas 1-5 bastan: GPU, Drive, repo, helpers, enlaces).
2. Vuelve a lanzar este lote con **el mismo** `RUN_NAME` o continúa con `03_entrega.ipynb`.
3. Los guiones `.story.json` ya guardados permiten reanudar sin volver a llamar al LLM.
